# Unified Dataset - Data Processing

Combines the processed traffic dataset (`traffic.csv`) with selected meteorological variables from `meteosat.csv`, weather forecast data from `open-meteo-weather-forecast.csv`, and sensor location/district info from `202468-283-intensidad-trafico.xlsx` into a single dataset.

## Specification

1. **Read** `dataset/processed/traffic.csv` into `df1`, `dataset/processed/meteosat.csv` into `df2`, `dataset/processed/open-meteo-weather-forecast.csv` into `df3`, and `dataset/processed/202468-283-intensidad-trafico.xlsx` into `df4`.
2. **Select** the meteorological columns of interest from `df2`: `prcp`, `temp`, `coco`, `wspd`, `cldc`.
3. **Merge** `df1` with the selected columns from `df2` on the `fecha` field (left join, preserving all traffic rows).
4. **Merge** the forecast columns from `df3` (`temperatura` as `forecast_temp`, `precipitacion` as `forecast_prec`) on the `fecha` field (left join).
5. **Merge** the sensor location columns from `df4` (`latitud`, `longitud`, `distrito`) on the `id` field (left join).
6. **Export** the combined DataFrame to `dataset/processed/traffic-meteo.csv`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Load datasets

In [2]:
df1 = pd.read_csv('dataset/processed/traffic.csv', parse_dates=['fecha'])
df2 = pd.read_csv('dataset/processed/meteosat.csv', parse_dates=['fecha'])
df3 = pd.read_csv('dataset/processed/open-meteo-weather-forecast.csv', parse_dates=['fecha'])
df4 = pd.read_excel('dataset/processed/202468-283-intensidad-trafico.xlsx')

print(f'Traffic:  {df1.shape[0]:,} rows, {df1.shape[1]} cols')
print(f'Meteosat: {df2.shape[0]:,} rows, {df2.shape[1]} cols')
print(f'Forecast: {df3.shape[0]:,} rows, {df3.shape[1]} cols')
print(f'Sensors:  {df4.shape[0]:,} rows, {df4.shape[1]} cols')

Traffic:  9,729,720 rows, 23 cols
Meteosat: 27,720 rows, 12 cols
Forecast: 27,720 rows, 3 cols
Sensors:  5,046 rows, 9 cols


In [3]:
display(df1.head())
display(df2.head())
display(df3.head())

,fecha,dia_semana,hora_dia,mes,festivo,vispera_festivo,id,tipo_elem,intensidad,ocupacion,...,n_error_E,n_error_S,n_error_NaN,error,hora_incompleta,hora_baja_confianza,intensidad_is_imputed,ocupacion_is_imputed,vmed_is_imputed,peso_loss
0,2023-01-01 00:00:00,6,0,1,1,0,1006,C30,1650.0,5.5,...,0,0,0,N,0,0,False,False,False,1.0
1,2023-01-01 01:00:00,6,1,1,1,0,1006,C30,240.0,1.0,...,0,0,0,N,0,0,False,False,False,1.0
2,2023-01-01 02:00:00,6,2,1,1,0,1006,C30,240.0,1.0,...,0,0,0,N,0,0,False,False,False,1.0
3,2023-01-01 03:00:00,6,3,1,1,0,1006,C30,240.0,1.0,...,0,0,0,N,0,0,False,False,False,1.0
4,2023-01-01 04:00:00,6,4,1,1,0,1006,C30,240.0,1.0,...,0,0,0,N,0,0,False,False,False,1.0


,fecha,temp,rhum,prcp,snwd,wdir,wspd,wpgt,pres,tsun,cldc,coco
0,2023-01-01 00:00:00,6.1,80,0.0,NaN,54,3.6,NaN,1027.0,NaN,1,1
1,2023-01-01 01:00:00,6.2,78,0.0,NaN,90,4.0,NaN,1026.6,NaN,2,1
2,2023-01-01 02:00:00,5.8,78,0.0,NaN,66,5.0,NaN,1026.5,NaN,1,2
3,2023-01-01 03:00:00,4.0,86,0.0,NaN,32,5.0,NaN,1026.4,NaN,2,2
4,2023-01-01 04:00:00,3.8,83,0.0,NaN,31,5.8,NaN,1025.8,NaN,1,2


,fecha,precipitacion,temperatura
0,2023-01-01 00:00:00,0.0,6.6
1,2023-01-01 01:00:00,0.0,6.1
2,2023-01-01 02:00:00,0.0,6.6
3,2023-01-01 03:00:00,0.0,6.3
4,2023-01-01 04:00:00,0.0,6.2


## 2. Merge traffic + meteorological variables

In [4]:
meteo_cols = ['fecha', 'prcp', 'temp', 'coco', 'wspd', 'cldc']
df = df1.merge(df2[meteo_cols], on='fecha', how='left')

forecast_cols = {'temperatura': 'forecast_temp', 'precipitacion': 'forecast_prec'}
df = df.merge(df3[['fecha'] + list(forecast_cols.keys())].rename(columns=forecast_cols), on='fecha', how='left')

sensor_cols = ['id', 'latitud', 'longitud', 'distrito']
df = df.merge(df4[sensor_cols], on='id', how='left')

print(f'Combined: {df.shape[0]:,} rows, {df.shape[1]} cols')
print(f'New columns: prcp, temp, coco, wspd, cldc, forecast_temp, forecast_prec, latitud, longitud, distrito')
print(f'\nNull counts in meteo + forecast + sensor columns:')
print(df[['prcp', 'temp', 'coco', 'wspd', 'cldc', 'forecast_temp', 'forecast_prec', 'latitud', 'longitud', 'distrito']].isnull().sum())

Combined: 9,729,720 rows, 33 cols
New columns: prcp, temp, coco, wspd, cldc, forecast_temp, forecast_prec, latitud, longitud, distrito

Null counts in meteo + forecast + sensor columns:
prcp             0
temp             0
coco             0
wspd             0
cldc             0
forecast_temp    0
forecast_prec    0
latitud          0
longitud         0
distrito         0
dtype: int64


In [5]:
df.head()

,fecha,dia_semana,hora_dia,mes,festivo,vispera_festivo,id,tipo_elem,intensidad,ocupacion,...,prcp,temp,coco,wspd,cldc,forecast_temp,forecast_prec,latitud,longitud,distrito
0,2023-01-01 00:00:00,6,0,1,1,0,1006,C30,1650.0,5.5,...,0.0,6.1,1,3.6,1,6.6,0.0,40.411894,-3.736324,10.0
1,2023-01-01 01:00:00,6,1,1,1,0,1006,C30,240.0,1.0,...,0.0,6.2,1,4.0,2,6.1,0.0,40.411894,-3.736324,10.0
2,2023-01-01 02:00:00,6,2,1,1,0,1006,C30,240.0,1.0,...,0.0,5.8,2,5.0,1,6.6,0.0,40.411894,-3.736324,10.0
3,2023-01-01 03:00:00,6,3,1,1,0,1006,C30,240.0,1.0,...,0.0,4.0,2,5.0,2,6.3,0.0,40.411894,-3.736324,10.0
4,2023-01-01 04:00:00,6,4,1,1,0,1006,C30,240.0,1.0,...,0.0,3.8,2,5.8,1,6.2,0.0,40.411894,-3.736324,10.0


In [6]:
df.isnull().sum()

fecha                    0
dia_semana               0
hora_dia                 0
mes                      0
festivo                  0
vispera_festivo          0
id                       0
tipo_elem                0
intensidad               0
ocupacion                0
vmed                     0
periodo_integracion      0
n_validos_15m            0
n_error_E                0
n_error_S                0
n_error_NaN              0
error                    0
hora_incompleta          0
hora_baja_confianza      0
intensidad_is_imputed    0
ocupacion_is_imputed     0
vmed_is_imputed          0
peso_loss                0
prcp                     0
temp                     0
coco                     0
wspd                     0
cldc                     0
forecast_temp            0
forecast_prec            0
latitud                  0
longitud                 0
distrito                 0
dtype: int64

In [7]:
# Verify that all 351 sensors in df match M30 or C30 in df4
merged_ids = df['id'].unique()
check = df4[df4['id'].isin(merged_ids)][['id', 'tipo_elem']].drop_duplicates()
bad = check[~check['tipo_elem'].isin(['M30', 'C30'])]

print(f'{len(merged_ids)} sensors in merged df, {len(check)} matched in df4')
if bad.empty:
    print('All sensors have tipo_elem M30 or C30.')
else:
    print(f'{len(bad)} sensor(s) with unexpected tipo_elem:')
    print(bad.to_string(index=False))

351 sensors in merged df, 351 matched in df4
91 sensor(s) with unexpected tipo_elem:
  id tipo_elem
6900     other
6899     other
6904     other
6905     other
6906     other
6901     other
6903     other
6887     other
6876     other
6874     other
6871     other
1020     other
6858     other
6857     other
6893     other
6878     other
1033     other
6848     other
6855     other
1034     other
1038     other
1037     other
6864     other
1031     other
1036     other
1035     other
6845     other
1040     other
1039     other
6835     other
1042     other
1011     other
6870     other
1019     other
1022     other
1027     other
7121     other
6856     other
1032     other
6854     other
6846     other
6844     other
6842     other
6840     other
6839     other
1041     other
1017     other
6837     other
1045     other
6881     other
6890     other
7120     other
6862     other
1043     other
6841     other
6827     other
6897     other
1050     other
6896     other
1052     other


## 2b. Compute angular position features (angle_sin, angle_cos)

In [8]:
# Build sensor-level table filtered to IDs present in df
active_ids = df['id'].unique()
sensor_coords = (
    df4[df4['id'].isin(active_ids)][['id', 'utm_x', 'utm_y']]
    .drop_duplicates(subset='id')
    .reset_index(drop=True)
)

# Count missing coordinates
n_total = len(sensor_coords)
missing_mask = sensor_coords['utm_x'].isna() | sensor_coords['utm_y'].isna()
n_missing = missing_mask.sum()
n_valid = n_total - n_missing

# Compute median center from valid sensors only
valid = sensor_coords[~missing_mask]
center_x = valid['utm_x'].median()
center_y = valid['utm_y'].median()

# Compute angle and circular features
dx = sensor_coords['utm_x'] - center_x
dy = sensor_coords['utm_y'] - center_y
angle = np.arctan2(dy, dx)
sensor_coords['angle_sin'] = np.sin(angle)
sensor_coords['angle_cos'] = np.cos(angle)

# Merge into df
df = df.merge(sensor_coords[['id', 'angle_sin', 'angle_cos']], on='id', how='left')

# Summary
print(f'Unique sensors processed: {n_total}')
print(f'  Valid coordinates:   {n_valid}')
print(f'  Missing coordinates: {n_missing}')
print(f'Estimated center: ({center_x:.2f}, {center_y:.2f})')
print(f'angle_sin range: [{df["angle_sin"].min():.6f}, {df["angle_sin"].max():.6f}]')
print(f'angle_cos range: [{df["angle_cos"].min():.6f}, {df["angle_cos"].max():.6f}]')

Unique sensors processed: 351
  Valid coordinates:   351
  Missing coordinates: 0
Estimated center: (441992.98, 4474314.02)
angle_sin range: [-1.000000, 0.999995]
angle_cos range: [-1.000000, 1.000000]


In [9]:
# Validation checks
valid_rows = df['angle_sin'].notna()

# Range checks
assert df.loc[valid_rows, 'angle_sin'].between(-1, 1).all(), 'angle_sin out of [-1, 1]'
assert df.loc[valid_rows, 'angle_cos'].between(-1, 1).all(), 'angle_cos out of [-1, 1]'

# sin^2 + cos^2 ≈ 1
identity = df.loc[valid_rows, 'angle_sin']**2 + df.loc[valid_rows, 'angle_cos']**2
assert (identity - 1.0).abs().max() < 1e-6, f'sin²+cos² deviates from 1 by {(identity - 1.0).abs().max()}'

# Constant within each sensor
per_sensor = df.groupby('id')[['angle_sin', 'angle_cos']].nunique()
assert (per_sensor <= 1).all().all(), 'angle_sin/angle_cos not constant within sensor'

print('All validation checks passed.')

All validation checks passed.


In [10]:
df.head(100000)

,fecha,dia_semana,hora_dia,mes,festivo,vispera_festivo,id,tipo_elem,intensidad,ocupacion,...,coco,wspd,cldc,forecast_temp,forecast_prec,latitud,longitud,distrito,angle_sin,angle_cos
0,2023-01-01 00:00:00,6,0,1,1,0,1006,C30,1650.0,5.50,...,1,3.6,1,6.6,0.0,40.411894,-3.736324,10.0,-0.128502,-0.991709
1,2023-01-01 01:00:00,6,1,1,1,0,1006,C30,240.0,1.00,...,1,4.0,2,6.1,0.0,40.411894,-3.736324,10.0,-0.128502,-0.991709
2,2023-01-01 02:00:00,6,2,1,1,0,1006,C30,240.0,1.00,...,2,5.0,1,6.6,0.0,40.411894,-3.736324,10.0,-0.128502,-0.991709
3,2023-01-01 03:00:00,6,3,1,1,0,1006,C30,240.0,1.00,...,2,5.0,2,6.3,0.0,40.411894,-3.736324,10.0,-0.128502,-0.991709
4,2023-01-01 04:00:00,6,4,1,1,0,1006,C30,240.0,1.00,...,2,5.8,1,6.2,0.0,40.411894,-3.736324,10.0,-0.128502,-0.991709
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,2024-12-02 11:00:00,0,11,12,0,0,1019,C30,3450.0,12.25,...,8,1.8,6,9.3,0.1,40.387345,-3.696709,12.0,-0.947276,-0.320420
99996,2024-12-02 12:00:00,0,12,12,0,0,1019,C30,3753.0,12.00,...,3,5.8,7,10.8,0.3,40.387345,-3.696709,12.0,-0.947276,-0.320420
99997,2024-12-02 13:00:00,0,13,12,0,0,1019,C30,3831.0,12.75,...,8,5.0,3,12.2,0.1,40.387345,-3.696709,12.0,-0.947276,-0.320420
99998,2024-12-02 14:00:00,0,14,12,0,0,1019,C30,4254.0,14.50,...,3,5.8,2,13.4,0.0,40.387345,-3.696709,12.0,-0.947276,-0.320420


## 3. Export

In [11]:
out_path = Path('dataset/processed/traffic-meteo.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_path, index=False)
print(f'Saved {len(df):,} rows to {out_path}')

Saved 9,729,720 rows to dataset/processed/traffic-meteo.csv
